# 02 · Ingestão dos documentos

Converte os PDFs e páginas do Refauna em **trechos** prontos para a busca:

1. lê o `documentos.csv` e confere se ele bate com os arquivos da pasta;
2. converte cada documento com o **Docling**, que preserva seções e tabelas;
3. fatia em trechos de até 450 tokens, sem cortar seções no meio, e descarta referências, agradecimentos e afins;
4. anexa a cada trecho os metadados do catálogo (título, autores, ano, espécie, acesso, fonte);
5. salva tudo em `dados/processado/trechos.jsonl`, no seu Drive.

Documentos com acesso `restrito` nunca entram. A GPU acelera o Docling, mas não é obrigatória.

**Antes de rodar:** confira se `PASTA_DADOS` aponta para a pasta do seu Drive que tem o `documentos.csv` e as subpastas `publico/` e `interno/`.

## 1. Instalação

In [ ]:
%%capture
!pip install -q docling

## 2. Preparação: Drive, repositório e módulo

In [ ]:
import os, subprocess, sys

# Pasta dos dados no seu Google Drive (a que tem o documentos.csv).
PASTA_DADOS = "/content/drive/MyDrive/refauna-slm/dados"
REPO_URL = "https://github.com/iamrodrigocoelho/refauna-slm"

from google.colab import drive
drive.mount("/content/drive")

if os.path.exists("refauna-slm"):
    subprocess.run(["git", "-C", "refauna-slm", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL], check=True)
sys.path.insert(0, "refauna-slm/src")

import importlib
import refauna_rag as rr
importlib.reload(rr)
print("Dados em:", PASTA_DADOS)

## 3. Catálogo

Se aparecer algum problema abaixo, corrija o `documentos.csv` ou os nomes dos arquivos antes de seguir:

- **faltando na pasta:** linha no CSV sem o arquivo correspondente (nome errado ou ainda não baixado);
- **sem linha no CSV:** arquivo na pasta que o CSV não descreve, e que por isso fica de fora;
- **pasta diferente do acesso:** arquivo em `publico/` marcado como `interno`, ou o contrário.

In [ ]:
catalogo = rr.carregar_catalogo(PASTA_DADOS)
problemas = rr.conferir_catalogo(catalogo, PASTA_DADOS)

print(f"{len(catalogo)} documentos no catálogo:", catalogo["acesso"].value_counts().to_dict())
for tipo, arquivos in problemas.items():
    if arquivos:
        print(f"\n⚠ {tipo.replace('_', ' ')}:")
        for a in arquivos:
            print("   ", a)
if not any(problemas.values()):
    print("Catálogo e pasta batem.")

## 4. Converter e fatiar

Na primeira execução o Docling baixa os modelos de layout e o tokenizador do BGE-M3; isso leva alguns minutos. Cada PDF científico leva de segundos a um minuto.

In [ ]:
import time

conversor = rr.criar_conversor()
fatiador = rr.criar_fatiador(max_tokens=450)

inicio = time.time()
trechos, relatorio = rr.ingerir(catalogo, PASTA_DADOS, conversor, fatiador)
print(f"{len(trechos)} trechos em {time.time() - inicio:.0f} s")
relatorio

## 5. Conferir os trechos

In [ ]:
import pandas as pd

tabela = pd.DataFrame(trechos)
tabela["palavras"] = tabela["texto"].str.split().str.len()
print("Palavras por trecho:", tabela["palavras"].describe().round(0).to_dict())
tabela.groupby(["arquivo", "acesso"]).size().rename("trechos").to_frame()

In [ ]:
# Dois trechos de exemplo, para ver se o texto saiu limpo.
for t in tabela.sample(min(2, len(tabela)), random_state=1).to_dict("records"):
    print(f"--- {t['arquivo']} · seção: {t['secao'] or '(sem seção)'}")
    print(t["texto"][:800], "\n")

Se aparecerem cabeçalhos de página, números de linha ou texto embaralhado, anote o arquivo em `docs/diario.md`: PDFs escaneados ou com duas colunas difíceis podem precisar de ajuste.

## 6. Salvar

In [ ]:
saida = f"{PASTA_DADOS}/processado/trechos.jsonl"
rr.salvar_trechos(trechos, saida)
print("Salvo em", saida)